# Notebook de Simulação e Teste para o LiveTrader (Multi-Ativo)

Este notebook permite executar o fluxo de trabalho do `LiveTrader` passo a passo para um **único ativo escolhido**, ideal para depuração e simulações rápidas.

**Pré-requisitos:**
1. O terminal MetaTrader 5 deve estar aberto e logado.
2. Os modelos de produção devem ter sido gerados pelo script `train_model.py` e estar na pasta `models/`.

In [ ]:
import pandas as pd
import yaml
import sys
from pathlib import Path
import MetaTrader5 as mt5

# Adiciona a pasta 'src' ao path para permitir as importações dos nossos módulos
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.live_trader import LiveTrader

### Passo 1: Escolha o Ativo e Inicialize o Trader

**Ação necessária:** Defina a variável `TICKER_PARA_TESTAR` com o nome do ativo que você deseja simular (deve ser um dos tickers configurados no `main.yaml`).

In [ ]:
# --- ESCOLHA O ATIVO AQUI ---
TICKER_PARA_TESTAR = "WDO$"  # Ex: "WINQ24" ou "SPY"
# ---------------------------

# Cria a instância do trader (ele carrega todas as configurações)
trader = LiveTrader(config_path='configs/main.yaml')

# Inicializa (conecta ao MT5 e carrega todos os modelos)
is_initialized = trader.initialize()

if is_initialized and TICKER_PARA_TESTAR in trader.asset_states:
    print(f"Trader inicializado com sucesso. Foco do teste: {TICKER_PARA_TESTAR}")
    # Armazena o estado do ativo específico que vamos testar
    asset_state_para_testar = trader.asset_states[TICKER_PARA_TESTAR]
else:
    print(f"Falha ao inicializar o trader ou o ticker '{TICKER_PARA_TESTAR}' não foi encontrado/carregado.")
    asset_state_para_testar = None

### Passo 2: Executar um Único Ciclo de Decisão (Single Tick)

Esta célula executa o corpo do loop `while` do robô uma única vez, **apenas para o ativo escolhido**. Você pode executar esta célula repetidamente para simular a passagem do tempo candle a candle.

In [ ]:
def run_single_tick(trader_instance, asset_state):
    """Executa um ciclo completo de busca de dados, predição e decisão para um único ativo."""
    if not asset_state:
        print("Estado do ativo não foi inicializado. Execute a célula anterior primeiro.")
        return

    ticker = asset_state['config']['ticker']
    ticker_order = asset_state['config']['live_trading']['ticker_order']
    timeframe_str = asset_state['config']['live_trading']['timeframe_str']
    mt5_timeframe = trader_instance._get_mt5_timeframe_from_string(timeframe_str)
    
    print(f"--- Executando um novo ciclo de decisão para: {ticker} ({timeframe_str}) ---")

    # 1. Buscar dados recentes
    print(f"Buscando os 300 candles mais recentes de {ticker}...")
    latest_data = trader_instance.provider.get_latest_rates(ticker, 300, mt5_timeframe)
    
    if latest_data.empty:
        print("Não foi possível obter dados recentes. Tentando conexão direta com o MT5...")        
        if not mt5.initialize():
            print("Falha ao conectar ao MetaTrader 5")            
            return
        
        print("Conectado ao MetaTrader 5 com sucesso")
        print(f"--- Buscando dados para o ativo {ticker} ({timeframe_str}) ---")

        rates = mt5.copy_rates_from_pos(trader_instance.ticker, mt5.TIMEFRAME_M5, 0, 300)
        latest_data = pd.DataFrame(rates)
        latest_data['time'] = pd.to_datetime(latest_data['time'], unit='s')
        latest_data.set_index('time', inplace=True)
        latest_data.rename(columns={'tick_volume': 'volume'}, inplace=True)

        mt5.shutdown()  # encerra a conexão com o MT5
    
    print(f"Último candle recebido: {latest_data.index[-1]}")
    display(latest_data.tail(3))

    # 2. Gerar features
    print("\nGerando features com os dados recentes...")
    featured_data = asset_state["strategy"].define_features(latest_data)
    X_live = featured_data[asset_state["strategy"].get_feature_names()].dropna()
    
    if X_live.empty:
        print("Não há dados suficientes para gerar features.")
        return

    # 3. Fazer a previsão (sinal)
    print("\nGerando sinal com o modelo de IA...")
    signal = asset_state["model"].predict(X_live)[-1]
    signal_text = 'COMPRA' if signal == 1 else 'VENDA'
    print(f"==> SINAL GERADO: {signal_text} ({signal}) ==<")

    # 4. Lógica de decisão (sugestão)
    print("\nAplicando lógica de decisão...")
    if asset_state["position"] is None:
        if signal == 1:
            trader_instance._execute_trade(ticker, 'BUY')
        elif signal == 0:
            trader_instance._execute_trade(ticker, 'SELL')
    else:
        print(f"Já existe uma posição aberta para {ticker} ({asset_state['position']}). Nenhuma nova ordem será enviada.")
        
    print("--- Ciclo de decisão concluído ---")


In [ ]:
# Executa a função para o nosso trader e o ativo escolhido
if is_initialized and asset_state_para_testar:
    run_single_tick(trader, asset_state_para_testar)

In [ ]:
# Novo run_single_tick
def run_single_tick(trader_instance, asset_state):
    if not asset_state:
        print("Estado do ativo não inicializado.")
        return

    data_ticker = asset_state['config']['ticker']
    timeframe_str = asset_state['config']['live_trading']['timeframe_str']
    mt5_timeframe = trader_instance._get_mt5_timeframe_from_string(timeframe_str)
    
    print(f"--- Executando ciclo de decisão para: {data_ticker} ({timeframe_str}) ---")

    # 1. Buscar dados recentes (usando o data_ticker)
    print(f"Buscando candles de {data_ticker}...")
    latest_data = trader_instance.provider.get_latest_rates(data_ticker, 300, mt5_timeframe)
    
    if latest_data.empty: return
    print(f"Último candle recebido: {latest_data.index[-1]}")
    display(latest_data.tail(3))

    # 2. Gerar features
    featured_data = asset_state["strategy"].define_features(latest_data)
    X_live = featured_data[asset_state["strategy"].get_feature_names()].dropna()
    if X_live.empty: 
        print("Dados insuficientes para gerar features.")
        return

    # 3. Gerar sinal
    print("\nGerando sinal com o modelo de IA...")
    signal = asset_state["model"].predict(X_live)[-1]
    signal_text = 'COMPRA' if signal == 1 else 'VENDA'
    print(f"==> SINAL GERADO: {signal_text} ({signal}) ==<")

    # 4. Lógica de decisão
    if asset_state["position"] is None:
        if signal == 1: trader_instance._execute_trade(data_ticker, 'BUY')
        elif signal == 0: trader_instance._execute_trade(data_ticker, 'SELL')
    else:
        print(f"Posição já aberta para {data_ticker} ({asset_state['position']}).")
        
    print("--- Ciclo de decisão concluído ---")

if is_initialized and asset_state_para_testar:
    run_single_tick(trader, asset_state_para_testar)

### Passo 3: Encerrar a Conexão

Ao final dos seus testes, execute esta célula para garantir que a conexão com o MetaTrader 5 seja encerrada corretamente.

In [ ]:
print("Encerrando conexão com o MetaTrader 5...")
mt5.shutdown()
print("Conexão encerrada.")